In [ ]:
import numpy as np
import os
from scipy.io import savemat
from scipy.fft import fft, fftshift
from dotenv import load_dotenv
from tqdm import tqdm

from rfml_uav.drone_rf.consts import BUI, M, Q
from rfml_uav.drone_rf.utils import get_segment_count

load_dotenv()


In [ ]:
load_path = os.path.join(os.getenv("DATA_PATH"), "raw")
save_path = os.path.join(os.getenv("DATA_PATH"), "PSD", "mat") 
os.makedirs(save_path, exist_ok=True)

L = int(1e5)  # Total number samples in a segment


def get_data(bui: str):
    data = []

    N = get_segment_count(bui)

    for n in tqdm(range(N)):
        # Loading raw CSV files
        x = np.loadtxt(os.path.join(load_path, f"{bui}L_{n}.csv"), delimiter=',')
        y = np.loadtxt(os.path.join(load_path, f"{bui}H_{n}.csv"), delimiter=',')
        
        # Re-segmenting and signal transformation
        for i in range(len(x) // L):
            st = i * L
            fi = (i + 1) * L
            xf = np.abs(fftshift(fft(x[st:fi] - np.mean(x[st:fi]), M)))[M//2:]
            yf = np.abs(fftshift(fft(y[st:fi] - np.mean(y[st:fi]), M)))[M//2:]
            c = np.mean(xf[-Q:]) / np.mean(yf[:Q]) # normalization factor
            segment_data = np.concatenate((xf, yf * c))
            data.append(segment_data)
    
    # Convert the list into a numpy array and square the data
    return np.array(data) ** 2

In [ ]:
# Main
for bui in BUI:
    Data = get_data(bui)
    
    # Saving data
    save_filepath = os.path.join(save_path, f"{bui}.mat")
    # An array is transposed to fit the .mat file convention
    savemat(save_filepath, {'Data': Data.T})